# 02 · Merge and clean

**Needs:** the four CSV files in `data/raw/`.
**Produces:** two merged datasets in `data/processed/`, saved as Parquet files.

| File | One row is | Use it to |
|---|---|---|
| `patents_2021-2026_by_field` | a publication selected for a field | compare fields |
| `patents_2021-2026_by_publication` | a distinct publication, with its fields in a list | count patents |

(A CSV version is not written here: `src/parquet_to_csv.py` creates it when you need it. See step 10.)

Steps: read and stack the files, parse dates and publication numbers, clean the text (keeping the original),
flag problematic abstracts, derive IPC columns, flag duplicates, build the second table, check and save.

**Rule of this notebook:** the files in `data/raw/` are never modified. The merged files contain the **cleaned** text only
(`title_clean`, `abstract_clean`); the original text stays available in `data/raw/`.

In [1]:
import html
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

## 1. Read and stack the four files

In [2]:
frames = [pd.read_csv(f, dtype=str) for f in sorted(RAW.glob("patents_*.csv"))]
df = pd.concat(frames, ignore_index=True)
df["wipo_field_number"] = df["wipo_field_number"].astype(int)
print(df.shape)

(64000, 8)


## 2. Dates and publication numbers

`publication_number` has three parts: office, number, kind code (`US-2022000739-A1`).
We split it. Some numbers contain more than one `-`, so the office is the first part and the kind code is the last.

In [3]:
df["publication_date"] = pd.to_datetime(df["publication_date"], format="%Y%m%d")
df["publication_year"] = df["publication_date"].dt.year
df["publication_month"] = df["publication_date"].dt.month

pn = df["publication_number"].str.split("-")
df["office"] = pn.str[0]
df["kind_code"] = pn.str[-1]

df[["publication_number", "office", "kind_code", "publication_date"]].head()

,publication_number,office,kind_code,publication_date
0,AU-2016364014-B2,AU,B2,2021-04-01
1,AU-2017269495-B2,AU,B2,2022-03-10
2,AU-2017291826-B2,AU,B2,2021-10-21
3,AU-2017356846-B2,AU,B2,2022-07-28
4,AU-2018213119-B2,AU,B2,2021-10-28


## 3. Clean the text

What we saw in the raw text:
- HTML entities, for example `user&#39;s` instead of `user's`;
- line breaks and repeated spaces inside abstracts, sometimes with numbers from a scanned page.

The cleaning is deliberately light: decode entities and collapse whitespace.
We do **not** lowercase, remove punctuation or fix OCR noise: these are choices for later analyses.

In [4]:
def clean_text(s: str) -> str:
    s = html.unescape(s)               # &#39; -> '
    s = re.sub(r"\s+", " ", s)         # line breaks and repeated spaces -> one space
    return s.strip()


df["title_clean"] = df["title_en"].map(clean_text)
df["abstract_clean"] = df["abstract_en"].map(clean_text)

Examples of what changed:

In [5]:
changed_t = df[df["title_en"] != df["title_clean"]]
changed_a = df[df["abstract_en"] != df["abstract_clean"]]
print("titles changed:   ", len(changed_t))
print("abstracts changed:", len(changed_a))

changed_t[["title_en", "title_clean"]].head(5)

titles changed:    270
abstracts changed: 7036


,title_en,title_clean
129,"Preparation method of 4,4&#39; -di (2-bromoacetyl) biphenyl","Preparation method of 4,4' -di (2-bromoacetyl) biphenyl"
149,5&#39; S-LNA nucleotides and oligonucleotides,5' S-LNA nucleotides and oligonucleotides
399,"Indolospiro [ benzofuran-2, 2&#39; -pyrrolidine ] compound, preparation method thereof and appli...","Indolospiro [ benzofuran-2, 2' -pyrrolidine ] compound, preparation method thereof and applicati..."
461,"Polycyclic compound for inhibiting Bruton&#39;s tyrosine kinase activity, pharmaceutical composi...","Polycyclic compound for inhibiting Bruton's tyrosine kinase activity, pharmaceutical composition..."
481,"Preparation method of high-purity single-crystal 2,2 &#39;-dimethyl-4, 4&#39; -diaminobiphenyl","Preparation method of high-purity single-crystal 2,2 '-dimethyl-4, 4' -diaminobiphenyl"


In [6]:
ex = changed_a.iloc[0]
print("BEFORE:", repr(ex["abstract_en"][:300]))
print("AFTER: ", repr(ex["abstract_clean"][:300]))

BEFORE: 'The present invention relates to a process for preparing a compound of formula [I], said process comprising the steps of: formula [II]+formula [III]−&gt;formula [I] (i) forming a reaction mixture comprising (a) a compound of formula [II], (b) a compound of formula [III] and (c) 1,2-propanediol or po'
AFTER:  'The present invention relates to a process for preparing a compound of formula [I], said process comprising the steps of: formula [II]+formula [III]−>formula [I] (i) forming a reaction mixture comprising (a) a compound of formula [II], (b) a compound of formula [III] and (c) 1,2-propanediol or polye'


## 4. Flag problematic abstracts

Some abstracts carry no information: placeholders such as `(無)`, "Information on abstract not available",
or a copyright line. Others are cut after a few words ("The present invention provides an").
We **flag** them, we do not delete them: the analyst decides what to do.

Two flags, with different meaning:
- `abstract_is_placeholder`: no information at all (strict rule, very few rows);
- `abstract_is_short`: fewer than 100 characters. A warning only: it includes truncated abstracts
  but also complete, very short ones ("Anti-CD94 antibodies and their use are described.").

In [7]:
PLACEHOLDER = re.compile(
    r"^[\W_]*[无無]?[\W_]*$"                 # empty, only punctuation, or "(无)" / "(無)"
    r"|abstract not available"
    r"|information on abstract"
    r"|^copyright",
    flags=re.IGNORECASE,
)
MIN_CHARS = 30      # abstracts shorter than this are treated as unusable
SHORT_CHARS = 100   # abstracts shorter than this get a warning flag

df["abstract_n_chars"] = df["abstract_clean"].str.len()
df["abstract_n_words"] = df["abstract_clean"].str.split().str.len()
df["title_n_words"] = df["title_clean"].str.split().str.len()

df["abstract_is_placeholder"] = df["abstract_clean"].str.contains(PLACEHOLDER) | (df["abstract_n_chars"] < MIN_CHARS)
df["abstract_is_short"] = df["abstract_n_chars"] < SHORT_CHARS
df["abstract_has_cjk"] = df["abstract_clean"].str.contains(r"[぀-ヿ一-鿿가-힯]")

print("placeholder or too short:", df["abstract_is_placeholder"].sum())
print("short (< 100 characters): ", df["abstract_is_short"].sum())
print("with Chinese/Japanese/Korean characters:", df["abstract_has_cjk"].sum())

df.loc[df["abstract_is_placeholder"], ["publication_number", "abstract_clean"]]

placeholder or too short: 3
short (< 100 characters):  95
with Chinese/Japanese/Korean characters: 5


,publication_number,abstract_clean
7483,JP-2026049542-A,"COPYRIGHT: (C)2026,JPO"
21771,EP-3828144-A1,Information on abstract not available
51923,TW-M646051-U,(無)


In [8]:
# a sample of the short abstracts: some are truncated, others are complete
df.loc[df["abstract_is_short"] & ~df["abstract_is_placeholder"], ["publication_number", "abstract_clean"]].sample(10, random_state=1)

,publication_number,abstract_clean
10861,CN-119013364-A,Provided herein is a two-part polyurethane adhesive composition.
25653,US-12114026-B2,A system for queuing flows to channels.
5746,US-2026125400-A1,"The present invention provides compounds, compositions thereof, and methods of using the same."
9866,US-2025185685-A1,The present invention relates to a coffee beverage and a method of manufacturing a coffee beverage.
31613,US-2021160454-A1,Systems and methods for implementing Soft-duplex functionality in an amplifier in a HFC network.
45772,US-12380769-B2,Disclosed herein is a gaming table with detectors for tracking a game.
47630,US-11779456-B2,Intraocular lens delivery devices and methods of use.
8125,CN-110191641-B,Novel lactonases are provided.
9456,EP-4681709-A1,Compositions for improving mental wellbeing and associated uses.
5333,SK-10441-Y1,Propolis extracts containing dimethyl sulfoxide and a method for their preparation are described.


## 5. IPC columns

`ipc_codes` is a text with codes separated by `;`. We keep it as it is and add:
- `n_ipc`: number of codes;
- `ipc_subclasses`: the distinct subclasses (first 4 characters, such as `G06N`);
- `n_subclasses`: how many distinct subclasses.

In [9]:
df["ipc_list"] = df["ipc_codes"].str.split(";")
df["n_ipc"] = df["ipc_list"].str.len()
df["ipc_subclasses"] = df["ipc_list"].map(lambda codes: sorted({c[:4] for c in codes}))
df["n_subclasses"] = df["ipc_subclasses"].str.len()
df = df.drop(columns="ipc_list")

df[["ipc_codes", "n_ipc", "ipc_subclasses", "n_subclasses"]].head()

,ipc_codes,n_ipc,ipc_subclasses,n_subclasses
0,C07C405/00,1,[C07C],1
1,A23D7/005;A23L33/17;A61K31/202;C07C51/00;C07C57/03;C11C1/02,6,"[A23D, A23L, A61K, C07C, C11C]",5
2,A61K31/422;A61K31/506;C07F9/09,3,"[A61K, C07F]",2
3,A23K20/26;A23K50/75;C07H21/04;C07K14/415;C12N9/16;C12N9/24,6,"[A23K, C07H, C07K, C12N]",4
4,A61K31/52;A61P35/00;C07D473/16,3,"[A61K, A61P, C07D]",3


## 6. Duplicates

The same text can appear in several publications, typically members of the same patent family
(a US and an EP publication of one invention). We flag them. The check is done on **distinct publications**,
so the 32 publications selected twice do not count as duplicates of themselves.

In [10]:
pubs_text = df.drop_duplicates("publication_number")
dup_title = pubs_text["title_clean"].str.lower().duplicated(keep=False)
dup_abstract = pubs_text["abstract_clean"].str.lower().duplicated(keep=False)

df["title_is_duplicate"] = df["publication_number"].isin(pubs_text.loc[dup_title, "publication_number"])
df["abstract_is_duplicate"] = df["publication_number"].isin(pubs_text.loc[dup_abstract, "publication_number"])

print("publications whose title appears more than once:   ", dup_title.sum())
print("publications whose abstract appears more than once:", dup_abstract.sum())

publications whose title appears more than once:    2153
publications whose abstract appears more than once: 167


Duplicate titles are mostly generic ("Display device"). Duplicate abstracts are much more likely to be the same invention.

## 7. Table 1: one row per (publication, field)

We add `n_fields` (in how many fields the publication was selected), then order the columns.
We keep the cleaned text only: keeping the raw text as well would double the size of the files.

In [11]:
df["n_fields"] = df.groupby("publication_number")["wipo_field_number"].transform("size")

by_field = df[
    [
        "wipo_sector", "wipo_field_number", "wipo_field_name",
        "publication_number", "office", "kind_code",
        "publication_date", "publication_year", "publication_month",
        "title_clean", "abstract_clean",
        "title_n_words", "abstract_n_chars", "abstract_n_words",
        "abstract_is_placeholder", "abstract_is_short", "abstract_has_cjk",
        "title_is_duplicate", "abstract_is_duplicate",
        "ipc_codes", "n_ipc", "ipc_subclasses", "n_subclasses",
        "n_fields",
    ]
].sort_values(["wipo_field_number", "publication_number"]).reset_index(drop=True)

by_field.shape

(64000, 24)

## 8. Table 2: one row per publication

The publications selected for two fields collapse into one row, with their fields in lists.
All other columns are identical in the two rows, so we take the first.

In [12]:
fields_agg = (
    by_field.sort_values("wipo_field_number")
    .groupby("publication_number")
    .agg(
        wipo_field_numbers=("wipo_field_number", list),
        wipo_field_names=("wipo_field_name", list),
        wipo_sectors=("wipo_sector", lambda s: sorted(set(s))),
    )
)

shared = [c for c in by_field.columns if c not in ("wipo_sector", "wipo_field_number", "wipo_field_name")]
by_publication = (
    by_field.drop_duplicates("publication_number")[shared]
    .merge(fields_agg, left_on="publication_number", right_index=True)
    .sort_values("publication_number")
    .reset_index(drop=True)
)

by_publication.shape

(63968, 24)

A publication selected for two fields:

In [13]:
by_publication[by_publication["n_fields"] > 1][["publication_number", "wipo_field_numbers", "wipo_field_names", "title_clean"]].head()

,publication_number,wipo_field_numbers,wipo_field_names,title_clean
2619,CN-110305820-B,"[15, 18]","[Biotechnology, Food chemistry]",Lactobacillus rhamnosus CCFM1064 and application thereof
3448,CN-111349264-B,"[17, 19]","[Macromolecular chemistry, polymers, Basic materials chemistry]",Hydrophobic oleophobic antistatic PVC composite film
5288,CN-112646191-A,"[17, 22]","[Macromolecular chemistry, polymers, Micro-structural and nano-technology]",Rare earth-organic framework nano fluorescent probe and preparation method and application thereof
5355,CN-112673133-A,"[28, 29]","[Textile and paper machines, Other special machines]",Molded body of cellulose fiber and method for producing same
7059,CN-113341486-A,"[19, 21]","[Basic materials chemistry, Surface technology, coating]",Structurally colored articles and methods of making and using structurally colored articles


## 9. Checks

We stop with an error if one of these numbers is not what we expect.

In [14]:
assert len(by_field) == 64_000
assert by_field.groupby("wipo_field_number").size().eq(2_000).all()
assert not by_field.duplicated(["publication_number", "wipo_field_number"]).any()
assert by_publication["publication_number"].is_unique
assert len(by_publication) == by_field["publication_number"].nunique()
assert by_publication["wipo_field_numbers"].str.len().sum() == len(by_field)
assert by_field["title_clean"].str.len().min() > 0 and by_field["abstract_clean"].str.len().min() > 0
print("all checks passed")
print("by field:", by_field.shape, "| by publication:", by_publication.shape)

all checks passed
by field: (64000, 24) | by publication: (63968, 24)


## 10. Save

We save **Parquet** files. Parquet is a file format for tables:
- it is about 2.5 times smaller than the same data in CSV (here 31 MB instead of 77 MB);
- it remembers the type of each column (dates stay dates, numbers stay numbers);
- it can hold lists, such as the list of fields of a publication;
- it is much faster to read.

A Parquet file is not a text file: you cannot open it in Excel or a text editor. If you need a CSV, for example to open the data
in Excel or to work on a small sample, run from the repository root:

```
python src/parquet_to_csv.py                          # both files, all rows
python src/parquet_to_csv.py --sample 5000            # 5,000 random rows
```

or, in Python, `df.to_csv("my_file.csv", index=False)`. See the header of `src/parquet_to_csv.py` for all options.

In [15]:
save_names = ["patents_2021-2026_by_field", "patents_2021-2026_by_publication"]

by_field.to_parquet(PROC / f"{save_names[0]}.parquet", index=False)
by_publication.to_parquet(PROC / f"{save_names[1]}.parquet", index=False)

for f in sorted(PROC.glob("patents_*.parquet")):
    print(f"{f.stat().st_size / 1e6:7.1f} MB   {f.name}")

   31.3 MB   patents_2021-2026_by_field.parquet
   32.5 MB   patents_2021-2026_by_publication.parquet


## Read the saved files back

The next notebooks start with these two lines.

In [16]:
check = pd.read_parquet(PROC / "patents_2021-2026_by_field.parquet")
print(check.shape)
check[["wipo_field_name", "publication_number", "publication_year", "ipc_subclasses"]].head(3)

(64000, 24)


,wipo_field_name,publication_number,publication_year,ipc_subclasses
0,"Electrical machinery, apparatus, energy",AU-2019295427-B2,2025,"[H02M, H05B, H05K]"
1,"Electrical machinery, apparatus, energy",AU-2019317728-B2,2024,[H05B]
2,"Electrical machinery, apparatus, energy",AU-2021404855-B9,2023,"[A23L, A47H, A61H, B05B, E04D, E04H, E06B, F21S, F24F, H01M, H02S]"


## Next

`03_eda_structure`: fields, years, offices and document types.